# Kimi-K3 MMLU-500 on a machine smaller than the model

Select a model and **Run all**. The GGUF stays on disk and `llama-server` lets the OS demand-page it through read-only mmap; no custom loader or external evaluation script is used. Point `MODEL_DIR` at local or mounted storage with room for all shards (594 GB for Unsloth or 330 GB for Neuron). The run will be very slow when the working set greatly exceeds RAM.

In [ ]:
MODEL = "unsloth"  # @param ["unsloth", "neuron"]
MODEL_DIR = "/content/models/kimi-k3"  # @param {type:"string"}

MODELS = {
    "unsloth": {
        "repo": "unsloth/Kimi-K3-GGUF",
        "revision": "a0836360ce58dfec088d966a97f2ddc8a606279b",
        "files": "UD-IQ1_S/*.gguf",
        "first": "UD-IQ1_S/Kimi-K3-UD-IQ1_S-00001-of-00014.gguf",
        "target": 412,
    },
    "neuron": {
        "repo": "vcruz305/Kimi-K3-Neuron-IQ1S-GGUF",
        "revision": "a2d6283870dd97d2f177c69d94fb18120e79fe65",
        "files": "*.gguf",
        "first": "k3-neuron-iq1s-00001-of-00009.gguf",
        "target": 342,
    },
}
spec = MODELS[MODEL]


In [ ]:
import os, subprocess, sys
from pathlib import Path

LLAMA_COMMIT = "23fac110127ba3ac56bd8b370eb0205a67564d55"
LLAMA = Path("/content/llama.cpp")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub[hf_xet]"], check=True)
if not (LLAMA / ".git").exists():
    subprocess.run(["git", "init", str(LLAMA)], check=True)
    subprocess.run(["git", "-C", str(LLAMA), "remote", "add", "origin", "https://github.com/unslothai/llama.cpp.git"], check=True)
subprocess.run(["git", "-C", str(LLAMA), "fetch", "--depth=1", "origin", LLAMA_COMMIT], check=True)
subprocess.run(["git", "-C", str(LLAMA), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run(["cmake", "-S", str(LLAMA), "-B", str(LLAMA / "build"), "-DGGML_CUDA=OFF", "-DLLAMA_BUILD_UI=OFF", "-DLLAMA_USE_PREBUILT_UI=OFF", "-DCMAKE_BUILD_TYPE=Release"], check=True)
subprocess.run(["cmake", "--build", str(LLAMA / "build"), "--target", "llama-server", "-j2"], check=True)

os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
from huggingface_hub import snapshot_download
token = os.environ.get("HF_TOKEN")
try:
    from google.colab import userdata
    token = token or userdata.get("HF_TOKEN")
except Exception:
    pass
snapshot_download(spec["repo"], revision=spec["revision"], allow_patterns=spec["files"], local_dir=MODEL_DIR, token=token)
MODEL_PATH = Path(MODEL_DIR) / spec["first"]


In [ ]:
import time, urllib.request

log = open("/content/llama-server.log", "w")
server = subprocess.Popen([
    str(LLAMA / "build/bin/llama-server"), "-m", str(MODEL_PATH),
    "-lm", "mmap", "--no-repack", "-ngl", "0", "--fit", "off",
    "--override-kv", "tokenizer.ggml.add_bos_token=bool:false,tokenizer.ggml.add_eos_token=bool:false",
    "-c", "1024", "-np", "1", "-cram", "0", "--no-warmup",
], stdout=log, stderr=subprocess.STDOUT)
while True:
    if server.poll() is not None:
        raise RuntimeError(Path("/content/llama-server.log").read_text()[-4000:])
    try:
        if urllib.request.urlopen("http://127.0.0.1:8080/health", timeout=2).status == 200:
            break
    except Exception:
        time.sleep(1)
print("llama-server ready")


In [ ]:
import hashlib, json, urllib.request

url = "https://raw.githubusercontent.com/my-other-github-account/banana-smasher/66f6e3143e30448cadf2ffb050413547bfd19171/notes/benchmarks/mmlu-density/mmlu500-v1/items.jsonl"
raw = urllib.request.urlopen(url).read()
assert hashlib.sha256(raw).hexdigest() == "df6704c4d02550b9155e106bc9a9e1bfe1164a663d509e41a76736bb60d01ded"
rows = [json.loads(line) for line in raw.splitlines()]
request = urllib.request.Request(
    "http://127.0.0.1:8080/completion",
    json.dumps({"prompt": [row["prompt"] for row in rows], "n_predict": 1, "n_probs": 100, "temperature": -1}).encode(),
    {"Content-Type": "application/json"},
)
answers = json.load(urllib.request.urlopen(request))
assert len(answers) == len(rows) == 500
correct = 0
for row, answer in zip(rows, answers, strict=True):
    scores = {x["id"]: x["logprob"] for x in answer["completion_probabilities"][0]["top_logprobs"]}
    correct += max(range(4), key=lambda i: scores[32 + i]) == row["answer_index"]
print(f"{correct}/500 ({correct / 5:.1f}%); published target: {spec['target']}/500")
